In [ ]:
!pip install -q torch==2.2.0 numpy==1.26.4 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install -q transformers==4.44.2 accelerate==0.33.0


In [ ]:
import torch
print("Versione torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
x = torch.rand(3).cuda()   
print("Test GPU OK:", x)

In [ ]:
import pickle

PATH_SOTTOGRAFO = "/kaggle/input/datasets/angelopaldino/datasetcora/cora_sottografo.pkl"
with open(PATH_SOTTOGRAFO, "rb") as f:
    sub = pickle.load(f)

testi = sub["testi"]
etichette = sub["etichette"]
grafo = sub["grafo"]
print(f"Nodi: {sub['n_nodi']}, Archi: {grafo.number_of_edges()}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

NOME_MODELLO = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    torch_dtype=torch.float16,
    device_map="cuda",
)

In [ ]:
import torch
import torch.nn.functional as F


class ModelloPerplexity:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    @torch.no_grad()
    def logprob_testo(self, testo_nodo, contesto=""):
        if contesto:
            prompt = contesto + "\n\n" + testo_nodo
            n_ctx_tok = self.tokenizer(contesto + "\n\n", return_tensors="pt").input_ids.shape[1]
        else:
            prompt = testo_nodo
            n_ctx_tok = 1

        ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        logits = self.model(ids).logits

        logprobs = F.log_softmax(logits[0, :-1], dim=-1)
        target = ids[0, 1:]
        tok_logprob = logprobs[torch.arange(target.shape[0]), target]

        nodo_logprob = tok_logprob[n_ctx_tok - 1:]
        somma = nodo_logprob.sum().item()
        n_token = nodo_logprob.shape[0]
        return somma, n_token

    def perplexity_testo(self, testo_nodo, contesto=""):
        somma, n_token = self.logprob_testo(testo_nodo, contesto)
        if n_token == 0:
            return float("inf")
        return float(torch.exp(torch.tensor(-somma / n_token)))

In [ ]:
m = ModelloPerplexity(model, tokenizer)

contesto = "Reinforcement learning is a branch of machine learning where agents learn by trial and error."
testo_coerente = "The agent maximizes a reward signal through interaction with the environment."
testo_incoerente = "The recipe requires two cups of flour and a pinch of salt."

print("PPL coerente:  ", round(m.perplexity_testo(testo_coerente, contesto), 2))
print("PPL incoerente:", round(m.perplexity_testo(testo_incoerente, contesto), 2))

In [ ]:
import random
import numpy as np
from tqdm import tqdm


class ScorerPerplexity:
    def __init__(self, modello, testi, max_nodi_contesto=5, max_char_testo=600, seed=42):
        self.modello = modello
        self.testi = testi
        self.max_nodi_contesto = max_nodi_contesto
        self.max_char_testo = max_char_testo
        self.rng = random.Random(seed)

    def _costruisci_contesto(self, nodo, membri_community):
        altri = [m for m in membri_community if m != nodo]
        if not altri:
            return ""
        k = min(self.max_nodi_contesto, len(altri))
        campione = self.rng.sample(altri, k)
        pezzi = [self.testi[m][:self.max_char_testo] for m in campione if self.testi[m]]
        return "\n\n".join(pezzi)

    def perplexity_nodo(self, nodo, membri_community):
        testo = self.testi[nodo][:self.max_char_testo]
        if not testo:
            return None
        contesto = self._costruisci_contesto(nodo, membri_community)
        return self.modello.perplexity_testo(testo, contesto)

    def ppl_partizione(self, partizione, mostra_avanzamento=True, descrizione="PPL(C)"):
        community = {}
        for nodo, c in enumerate(partizione):
            community.setdefault(c, []).append(nodo)

        ppl_per_nodo = {}
        iteratore = range(len(partizione))
        if mostra_avanzamento:
            iteratore = tqdm(iteratore, desc=descrizione, unit="nodo")

        for nodo in iteratore:
            c = partizione[nodo]
            ppl = self.perplexity_nodo(nodo, community[c])
            if ppl is not None:
                ppl_per_nodo[nodo] = ppl

        if not ppl_per_nodo:
            return float("inf"), {}
        media = float(np.mean(list(ppl_per_nodo.values())))
        return media, ppl_per_nodo

In [ ]:
scorer = ScorerPerplexity(m, testi)

print("Calcolo PPL(C) con ground-truth...")
ppl_gt, _ = scorer.ppl_partizione(etichette, descrizione="PPL ground-truth")

print("Calcolo PPL(C) con partizione casuale...")
rng = np.random.default_rng(0)
part_random = rng.integers(0, len(set(etichette)), size=len(etichette))
ppl_rand, _ = scorer.ppl_partizione(part_random, descrizione="PPL casuale")

print(f"\nPPL(C) ground-truth: {ppl_gt:.3f}")
print(f"PPL(C) casuale:      {ppl_rand:.3f}")

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelopaldino/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
G = nx.Graph()
G.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        G.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette dal CSV combinato, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)

testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")

etichette = tape["label"].values.astype(int)
print(f"Nodi: {n_nodi}, Archi: {G.number_of_edges()}, Classi: {len(set(etichette))}")

In [ ]:
testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(testi_per_embedding, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
print("Shape embeddings:", embeddings.shape)  # (2708, 384)

In [ ]:
scorer = ScorerPerplexity(m, testi, embeddings)

print("PPL(C) ground-truth sul grafo completo...")
ppl_gt, ppl_nodi_gt = scorer.ppl_partizione(etichette, descrizione="PPL ground-truth")

print("PPL(C) casuale sul grafo completo...")
rng = np.random.default_rng(0)
part_random = rng.integers(0, len(set(etichette)), size=len(etichette))
ppl_rand, _ = scorer.ppl_partizione(part_random, descrizione="PPL casuale")

print(f"\nPPL(C) ground-truth: {ppl_gt:.3f}")
print(f"PPL(C) casuale:      {ppl_rand:.3f}")
print(f"Divario: {ppl_rand - ppl_gt:.3f} ({100*(ppl_rand-ppl_gt)/ppl_gt:.1f}%)")

In [ ]:
import json
risultati = {
    "ppl_ground_truth": ppl_gt,
    "ppl_casuale": ppl_rand,
    "divario_assoluto": ppl_rand - ppl_gt,
    "divario_percentuale": 100*(ppl_rand-ppl_gt)/ppl_gt,
    "n_nodi": n_nodi,
}
with open("/kaggle/working/risultati_grafo_completo.json", "w") as f:
    json.dump(risultati, f, indent=2)

# salvo anche le perplessita' per nodo (utili per la relazione)
np.save("/kaggle/working/ppl_per_nodo_gt.npy", np.array([ppl_nodi_gt.get(i, np.nan) for i in range(n_nodi)]))
print("Risultati salvati in /kaggle/working/")